# Structured Output 结构化输出

结构化输出允许代理以特定且可预测的格式返回数据。

无需解析自然语言响应，即可获得 JSON 对象、 Pydantic 模型或数据类形式的结构化数据，您的应用程序可以直接使用这些数据。

LangChain[`create_agent`](https://reference.langchain.com/python/langchain/agents/factory/create_agent)能够自动处理结构化输出。用户设置所需的结构化输出模式，当模型生成结构化数据时，这些数据会被捕获、验证，并以`'structured_response'`代理状态键的形式返回。

```python
def create_agent(
    ...
    response_format: Union[
        ToolStrategy[StructuredResponseT],
        ProviderStrategy[StructuredResponseT],
        type[StructuredResponseT],
        None,
    ]
```

##  Response format 响应格式

用于`response_format`控制代理如何返回结构化数据：

- **`ToolStrategy[StructuredResponseT]`**使用工具调用进行结构化输出
- **`ProviderStrategy[StructuredResponseT]`**使用提供商原生结构化输出
- **`type[StructuredResponseT]`**模式类型 - 根据模型功能自动选择最佳策略
- **`None`**未明确请求结构化输出

`structured_response`结构化响应以Agent最终状态的键返回。

### ProviderStrategy 供应商策略

#### Schema

定义结构化输出格式的模式。支持：

- **Pydantic 模型**：`BaseModel`带有字段验证的子类。返回已验证的 Pydantic 实例。
- **数据类**：带有类型注解的 Python 数据类。返回字典。
- **TypedDict**：类型化字典类。返回字典。
- **JSON Schema**：包含 JSON 模式规范的字典。返回字典。

#### Strict

`ProviderStrategy`当您直接传递模式类型[`create_agent.response_format`](https://reference.langchain.com/python/langchain/agents/factory/create_agent)且模型支持原生结构化输出时， LangChain 会自动使用：

##### 示例代码

###### Pydantic

In [ ]:
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

model = init_chat_model(
    model="gemma4:e4b",
    model_provider="ollama",
)
agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])

e:\code\LangChainDemo\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
e:\code\LangChainDemo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


name='John Doe' email='john@example.com' phone='(555) 123-4567'


###### dataclass

In [2]:
from dataclasses import dataclass
from langchain.agents import create_agent


@dataclass
class ContactInfo:
    """Contact information for a person."""

    name: str  # The name of the person
    email: str  # The email address of the person
    phone: str  # The phone number of the person


model = init_chat_model(
    model="gemma4:e4b",
    model_provider="ollama",
)
tools = []
agent = create_agent(
    model=model,
    tools=tools,
    response_format=ContactInfo,  # Auto-selects ProviderStrategy
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567",
            }
        ]
    }
)

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

###### TypedDict

In [3]:
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""

    name: str  # The name of the person
    email: str  # The email address of the person
    phone: str  # The phone number of the person


model = init_chat_model(
    model="gemma4:e4b",
    model_provider="ollama",
)
tools = []
agent = create_agent(
    model=model,
    tools=tools,
    response_format=ContactInfo,  # Auto-selects ProviderStrategy
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567",
            }
        ]
    }
)

result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

###### JSON

In [4]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy

contact_info_schema = {
    "type": "object",
    "description": "Contact information for a person.",
    "properties": {
        "name": {"type": "string", "description": "The name of the person"},
        "email": {"type": "string", "description": "The email address of the person"},
        "phone": {"type": "string", "description": "The phone number of the person"},
    },
    "required": ["name", "email", "phone"],
}
model = init_chat_model(
    model="gemma4:e4b",
    model_provider="ollama",
)
tools = []
agent = create_agent(
    model=model, tools=tools, response_format=ProviderStrategy(contact_info_schema)
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567",
            }
        ]
    }
)

result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}